# Práctica #5 — Árboles de Derivación y Análisis Sintáctico

---

## Integrantes

* ALAN ENRIQUE CHALA PEREA
* SAMUEL VELASQUEZ BERRIO
* JORGE LUIS RODRIGUEZ JIMENEZ

---

## Objetivos

Desarrollar un programa en Python que, dada una cadena ingresada por teclado, determine si pertenece al lenguaje generado por la gramática:

```
S  →  aAb  |  aBb  |  λ
A  →  aAb  |  a
B  →  aBb  |  b  |  λ
```

En caso afirmativo se construye y renderiza gráficamente el árbol de derivación. En caso contrario se lanza un mensaje de error.

---

## Cuerpo del Informe

La clase `Nodo` representa cada elemento del árbol (etiqueta, hijos, regla aplicada). La clase `Parser` actúa como el motor del análisis sintáctico mediante tres métodos recursivos:

* `parse_S`: Estado inicial — intenta `S → aAb`, luego `S → aBb`, luego `S → λ`.
* `parse_A`: Intenta `A → aAb` con backtracking; si falla aplica `A → a`.
* `parse_B`: Intenta `B → aBb`, luego `B → b`, finalmente `B → λ`.

La visualización del árbol se realiza con `matplotlib` calculando posiciones mediante un algoritmo de layout recursivo que distribuye los nodos horizontalmente según su subárbol.

---

## Conclusiones

El parser recursivo descendente refleja directamente la estructura de la gramática, haciendo el código intuitivo y mantenible. El backtracking es esencial cuando varias producciones comparten prefijos, como `S → aAb` y `S → aBb`. El lenguaje generado es más restringido de lo que parece: la rama B efectivamente solo produce `abb`, mientras que la rama A genera el patrón `a^(n+1) b^n` para `n ≥ 1`. La visualización con `matplotlib` permite verificar visualmente cada derivación, facilitando la comprensión del proceso de análisis sintáctico.

---
## Instalación de dependencias

In [ ]:
!pip install matplotlib -q

## Implementación del Parser y Árbol de Derivación

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─── Nodo del árbol ──────────────────────────────────────────────────────────
class Nodo:
    def __init__(self, etiqueta, terminal=False, regla=None):
        self.etiqueta = etiqueta
        self.terminal = terminal
        self.regla    = regla
        self.hijos    = []

    def agregar_hijo(self, hijo):
        self.hijos.append(hijo)


# ─── Parser recursivo descendente ────────────────────────────────────────────
class Parser:
    def __init__(self, cadena):
        self.entrada = list(cadena)
        self.pos = 0

    def parse_S(self):
        nodo = Nodo('S')
        if self.pos >= len(self.entrada):
            nodo.regla = 'S → λ'
            nodo.agregar_hijo(Nodo('λ', terminal=True))
            return nodo
        if self.entrada[self.pos] == 'a':
            self.pos += 1
            pos_g = self.pos
            # S → aAb
            hijo_A = self.parse_A()
            if hijo_A and self.pos < len(self.entrada) and self.entrada[self.pos] == 'b':
                self.pos += 1
                nodo.regla = 'S → aAb'
                nodo.agregar_hijo(Nodo('a', terminal=True))
                nodo.agregar_hijo(hijo_A)
                nodo.agregar_hijo(Nodo('b', terminal=True))
                return nodo
            # S → aBb
            self.pos = pos_g
            hijo_B = self.parse_B()
            if hijo_B and self.pos < len(self.entrada) and self.entrada[self.pos] == 'b':
                self.pos += 1
                nodo.regla = 'S → aBb'
                nodo.agregar_hijo(Nodo('a', terminal=True))
                nodo.agregar_hijo(hijo_B)
                nodo.agregar_hijo(Nodo('b', terminal=True))
                return nodo
            self.pos -= 1
        return None

    def parse_A(self):
        nodo = Nodo('A')
        if self.pos < len(self.entrada) and self.entrada[self.pos] == 'a':
            self.pos += 1
            pos_g = self.pos
            # A → aAb
            hijo_A = self.parse_A()
            if hijo_A and self.pos < len(self.entrada) and self.entrada[self.pos] == 'b':
                self.pos += 1
                nodo.regla = 'A → aAb'
                nodo.agregar_hijo(Nodo('a', terminal=True))
                nodo.agregar_hijo(hijo_A)
                nodo.agregar_hijo(Nodo('b', terminal=True))
                return nodo
            # A → a
            self.pos = pos_g
            nodo.regla = 'A → a'
            nodo.agregar_hijo(Nodo('a', terminal=True))
            return nodo
        return None

    def parse_B(self):
        nodo = Nodo('B')
        pos_g = self.pos
        # B → aBb
        if self.pos < len(self.entrada) and self.entrada[self.pos] == 'a':
            self.pos += 1
            hijo_B = self.parse_B()
            if hijo_B and self.pos < len(self.entrada) and self.entrada[self.pos] == 'b':
                self.pos += 1
                nodo.regla = 'B → aBb'
                nodo.agregar_hijo(Nodo('a', terminal=True))
                nodo.agregar_hijo(hijo_B)
                nodo.agregar_hijo(Nodo('b', terminal=True))
                return nodo
            self.pos = pos_g
        # B → b
        if self.pos < len(self.entrada) and self.entrada[self.pos] == 'b':
            self.pos += 1
            nodo.regla = 'B → b'
            nodo.agregar_hijo(Nodo('b', terminal=True))
            return nodo
        # B → λ
        nodo.regla = 'B → λ'
        nodo.agregar_hijo(Nodo('λ', terminal=True))
        return nodo

    def analizar(self):
        self.pos = 0
        arbol = self.parse_S()
        if arbol and self.pos == len(self.entrada):
            return arbol
        return None


# ─── Layout del árbol ─────────────────────────────────────────────────────────
def calcular_posiciones(nodo, profundidad=0, contador=[0]):
    """Asigna (x, y) a cada nodo en el árbol."""
    if not nodo.hijos:
        nodo.x = contador[0]
        nodo.y = -profundidad
        contador[0] += 1
        return
    for hijo in nodo.hijos:
        calcular_posiciones(hijo, profundidad + 1, contador)
    nodo.x = sum(h.x for h in nodo.hijos) / len(nodo.hijos)
    nodo.y = -profundidad


def dibujar_arbol(nodo, ax):
    """Dibuja recursivamente nodos y aristas."""
    COLORES = {
        'S': '#4C4AE8',
        'A': '#E07B00',
        'B': '#C0392B',
        'a': '#1A7A4A',
        'b': '#1A7A4A',
        'λ': '#888888',
    }
    color = COLORES.get(nodo.etiqueta, '#555555')

    for hijo in nodo.hijos:
        ax.plot([nodo.x, hijo.x], [nodo.y, hijo.y],
                color='#BBBBBB', linewidth=1.5, zorder=1)
        dibujar_arbol(hijo, ax)

    circle = plt.Circle((nodo.x, nodo.y), 0.35,
                         color=color, zorder=2)
    ax.add_patch(circle)
    ax.text(nodo.x, nodo.y, nodo.etiqueta,
            ha='center', va='center', fontsize=11,
            fontweight='bold', color='white', zorder=3)


def recolectar_reglas(nodo, reglas=None):
    if reglas is None:
        reglas = []
    if nodo.regla:
        reglas.append(nodo.regla)
    for h in nodo.hijos:
        recolectar_reglas(h, reglas)
    return reglas


def visualizar(cadena):
    parser = Parser(cadena)
    arbol  = parser.analizar()

    if not arbol:
        print(f'✗  "{cadena}" NO es aceptada por la gramática.')
        return

    reglas = recolectar_reglas(arbol)
    print(f'✓  "{cadena or "λ"}" es aceptada')
    print('   Reglas:', '  →  '.join(reglas))

    calcular_posiciones(arbol, contador=[0])

    # Calcular bounding box
    todos = []
    def recolectar(n):
        todos.append(n)
        for h in n.hijos: recolectar(h)
    recolectar(arbol)

    xs = [n.x for n in todos]
    ys = [n.y for n in todos]
    margen = 0.8
    ancho  = max(6, max(xs) - min(xs) + 2)
    alto   = max(4, max(ys) - min(ys) + 2)

    fig, ax = plt.subplots(figsize=(ancho * 0.9, alto * 1.1))
    ax.set_xlim(min(xs) - margen, max(xs) + margen)
    ax.set_ylim(min(ys) - margen, max(ys) + margen)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_facecolor('#F8F9FA')
    fig.patch.set_facecolor('#F8F9FA')

    ax.set_title(f'Árbol de derivación: "{cadena or "λ"}"',
                 fontsize=13, fontweight='bold', pad=12, color='#222222')

    dibujar_arbol(arbol, ax)

    # Leyenda
    leyenda = [
        mpatches.Patch(color='#4C4AE8', label='S  (símbolo inicial)'),
        mpatches.Patch(color='#E07B00', label='A  (no terminal)'),
        mpatches.Patch(color='#C0392B', label='B  (no terminal)'),
        mpatches.Patch(color='#1A7A4A', label='a, b  (terminales)'),
        mpatches.Patch(color='#888888', label='λ  (vacía)'),
    ]
    ax.legend(handles=leyenda, loc='lower right',
              fontsize=8, framealpha=0.8)

    plt.tight_layout()
    plt.show()

print('Parser listo. Usa visualizar("cadena") para analizar.')

## Ejemplos de cadenas aceptadas

In [ ]:
visualizar('aab')

In [ ]:
visualizar('abb')

In [ ]:
visualizar('aaabb')

In [ ]:
visualizar('aaaabbb')

In [ ]:
visualizar('')  # cadena vacía λ

## Ejemplos de cadenas rechazadas

In [ ]:
visualizar('ba')

In [ ]:
visualizar('aabb')

In [ ]:
visualizar('abc')

## Ingresar cadena manualmente

In [ ]:
cadena = input('Ingresa una cadena (Enter para λ): ')
visualizar(cadena)